# Time Series & Retail Forecasting

- [Time Series](#Time-Series)
  - [Decomposition: trend + seasonality + residual](#Decomposition:-trend-+-seasonality-+-residual)
  - [Autocorrelation (ACF), worked by hand](#Autocorrelation-(ACF),-worked-by-hand)
  - [Stationarity and differencing, worked by hand](#Stationarity-and-differencing,-worked-by-hand)
  - [Exponential smoothing, worked by hand](#Exponential-smoothing,-worked-by-hand)
  - [ARIMA](#ARIMA)
- [Retail Demand Forecasting at Scale](#Retail-Demand-Forecasting-at-Scale)
  - [One global model for many SKUs](#One-global-model-for-many-SKUs)
  - [Forecast a distribution, then decide](#Forecast-a-distribution,-then-decide)
  - [Sales are not demand](#Sales-are-not-demand)
  - [Time-series cross-validation](#Time-series-cross-validation)
  - [Forecast metrics: MAPE, WAPE and MASE](#Forecast-metrics:-MAPE,-WAPE-and-MASE)
- [Blogs](#Blogs)
  - [Pitfalls](#Pitfalls)

### Time Series

Time series data is indexed by time, and its observations are not independent. This breaks the IID assumption most other models rely on (see the "independence of errors" assumption in the Linear Regression section of `08-classical-ml.ipynb`); here it is violated by construction. This notebook covers decomposition, autocorrelation, stationarity, exponential smoothing and ARIMA.

#### Decomposition: trend + seasonality + residual

The additive model is y = trend + seasonal + residual. The example below is built from known pieces to show how they combine. Real decomposition works backward from the raw y to recover them (done in the code below).
```
trend (linear, +1 per step):     [10, 11, 12, 13, 14, 15, 16, 17]
seasonal (period 4, repeating):  [+2, +3,  0, -3, +2, +3,  0, -3]
residual:                        [ 0,  0,  0,  0,  0,  0,  0,  0]  (clean construction, no noise)

y = trend + seasonal:            [12, 14, 12, 10, 16, 18, 16, 14]
```
The seasonal pattern repeats identically every 4 steps (+2, +3, 0, -3) while the trend keeps climbing underneath it, and the observed series y is their sum. Real decomposition (statsmodels' `seasonal_decompose`, used below) estimates the trend with a moving average over one full season, estimates the seasonal component from what is left after removing the trend, and treats the remainder as the residual.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

trend = np.arange(10, 18)
seasonal = np.tile([2, 3, 0, -3], 2)
y = trend + seasonal

series = pd.Series(y, index=pd.date_range("2026-01-01", periods=8, freq="D"))
result = seasonal_decompose(series, model="additive", period=4, extrapolate_trend="freq")

print("recovered trend:\n", result.trend.values)
print("recovered seasonal:\n", result.seasonal.values)

#### Autocorrelation (ACF), worked by hand

Autocorrelation is the correlation of a series with a lagged copy of itself: does y_t tell you something about y_(t+1), or further out? The lag-1 formula is ACF(1) = sum(dev_t * dev_(t+1)) / sum(dev_t^2), where dev_t = y_t - mean(y).

Worked example, series [1, 3, 2, 4, 3, 5], mean = 3:
```
deviations: [-2, 0, -1, 1, 0, 2]

lag-1 products: (-2*0), (0*-1), (-1*1), (1*0), (0*2) = 0, 0, -1, 0, 0
sum of products = -1
sum of squared deviations = 4+0+1+1+0+4 = 10

ACF(1) = -1/10 = -0.1
```
This is weak negative autocorrelation at lag 1: today's value gives almost no useful signal about tomorrow's in this toy series. A strongly autocorrelated series (ACF near +1 or -1 at some lag) is what the AR models below exploit. If y_t reliably predicts y_(t+1), a model can be built directly on that relationship.

In [ ]:
from statsmodels.tsa.stattools import acf

toy_series = np.array([1, 3, 2, 4, 3, 5])
acf_values = acf(toy_series, nlags=2, fft=False)
print("ACF at lags 0,1,2:", acf_values.round(3))

#### Stationarity and differencing, worked by hand

A stationary series has a roughly constant mean and variance over time, with no trend and no growing or shrinking spread. ARIMA (below) assumes stationarity, and a series with an obvious trend breaks that assumption because its mean keeps changing.

Differencing removes a trend: y'_t = y_t - y_(t-1). Worked example, a perfectly linear trend [10, 13, 16, 19, 22] (each step +3):
```
differenced: [13-10, 16-13, 19-16, 22-19] = [3, 3, 3, 3]
```
The differenced series is constant, so its mean and variance no longer change: it is stationary by construction. This is the "I" (integrated) in ARIMA, the number of times you difference the raw series before the AR and MA parts can work.

In [ ]:
linear_trend_series = np.array([10, 13, 16, 19, 22])
differenced = np.diff(linear_trend_series)
print("differenced series:", differenced, "(constant -> stationary)")

#### Exponential smoothing, worked by hand

Formula: S_t = alpha*y_t + (1-alpha)*S_(t-1). It is a weighted average that favors recent observations without discarding older ones, which are baked into S_(t-1).

Worked example, series [10, 12, 11, 15], alpha = 0.3, S_0 = y_0 = 10:
```
S_1 = 0.3*12 + 0.7*10 = 3.6 + 7.0 = 10.6
S_2 = 0.3*11 + 0.7*10.6 = 3.3 + 7.42 = 10.72
S_3 = 0.3*15 + 0.7*10.72 = 4.5 + 7.504 = 12.004
```
The smoothed series [10, 10.6, 10.72, 12.0] lags behind and dampens the raw swings [10, 12, 11, 15]. It is a cheap baseline forecast: the next forecast is the latest S_t carried forward.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

series_smooth = np.array([10, 12, 11, 15], dtype=float)
model = SimpleExpSmoothing(series_smooth).fit(smoothing_level=0.3, optimized=False)
print("fitted (smoothed) values:", model.fittedvalues.round(3))

#### ARIMA

**AR(p), autoregressive:** predict y_t from its own past p values. For AR(1), y_t = c + phi*y_(t-1) + error_t. The coefficient phi is the same kind of learned weight as in linear regression (see `08-classical-ml.ipynb`), applied to the series' own lagged values as features.

**I(d), integrated:** the number of times to difference the series (see Stationarity above) before fitting, so that it becomes stationary.

**MA(q), moving average:** predict y_t from the past q forecast errors, not past values: y_t = c + theta*error_(t-1) + error_t. It models the idea that a recent surprise (a large forecast error) has a lingering effect on the next observation.

ARIMA(p,d,q) combines all three: difference d times, then fit an AR(p) + MA(q) model to the differenced series. p, d and q are usually chosen from ACF/PACF (partial autocorrelation) plots, or by automated search over small combinations, similar to grid search in `06-metrics-selection-tuning.ipynb` but minimizing an information criterion such as AIC instead of a validation score.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

rng = np.random.default_rng(0)
arima_series = np.cumsum(rng.normal(size=50)) + 50  # a random-walk-with-drift-like series

model = ARIMA(arima_series, order=(1, 1, 1))  # p=1, d=1, q=1
fitted = model.fit()
print(fitted.summary().tables[1])

forecast = fitted.forecast(steps=3)
print("\n3-step forecast:", forecast)

### Retail Demand Forecasting at Scale

The sections above forecast one series at a time. A retailer forecasts tens of thousands of SKUs, and the forecast feeds an ordering decision, so the problem changes in three ways: use one model across all series, forecast a distribution instead of a point, and remember that sales are not the same as demand.

#### One global model for many SKUs

A separate ARIMA per SKU does not scale to 10,000+ items and cannot share what it learns across similar items. Common practice is one gradient-boosted model over all series, with lag features (sales 1, 4 and 52 weeks ago), rolling means, calendar position, promotion flags and static attributes, trained on a time-based split. Compare it with simple baselines (last week, same week last year) and report WAPE (sum of absolute errors divided by sum of actuals), since MAPE explodes on low-volume SKUs.

Synthetic weekly sales for 60 SKUs with seasonality, trend and promotions:

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

rng = np.random.default_rng(0)
n_sku, n_weeks = 60, 156
rows = []
for s in range(n_sku):
    level, trend = rng.gamma(3, 8), rng.normal(0, 0.03)
    amp, phase = rng.uniform(0.1, 0.5), rng.uniform(0, 2 * np.pi)
    promo = rng.random(n_weeks) < 0.08
    for w in range(n_weeks):
        mu = level * (1 + amp * np.sin(2 * np.pi * w / 52 + phase)) * (1 + trend * w / 52) * (1.6 if promo[w] else 1.0)
        rows.append((s, w, rng.poisson(max(mu, 0.1)), int(promo[w])))
df = pd.DataFrame(rows, columns=["sku", "week", "sales", "promo"]).sort_values(["sku", "week"])

g = df.groupby("sku").sales
df["lag1"] = g.shift(1); df["lag4"] = g.shift(4); df["lag52"] = g.shift(52)
df["roll4"] = g.transform(lambda x: x.shift(1).rolling(4).mean())
df["week_of_year"] = df.week % 52
features = ["lag1", "lag4", "lag52", "roll4", "week_of_year", "promo"]

train, test = df[df.week < 130].dropna(subset=["lag52"]), df[df.week >= 130]
gbm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, random_state=0).fit(train[features], train.sales)

def wape(actual, pred): return np.abs(actual - pred).sum() / actual.sum()
print(f"WAPE last week     : {wape(test.sales, test.lag1):.3f}")
print(f"WAPE same week LY  : {wape(test.sales, test.lag52):.3f}")
print(f"WAPE global GBM    : {wape(test.sales, gbm.predict(test[features])):.3f}")

WAPE last week     : 0.272
WAPE same week LY  : 0.265
WAPE global GBM    : 0.191


The single model learns the seasonal shape and the promotion lift from all SKUs at once and beats both baselines. In production the lag features for the forecast horizon come from the forecasts themselves (or the horizon is forecast directly), which this one-step-ahead demo skips.

#### Forecast a distribution, then decide

Ordering is asymmetric. A unit short loses the margin, a unit left over costs storage or a markdown. The newsvendor result says to order the quantile of the demand distribution at the critical ratio Cu / (Cu + Co), where Cu is the cost of one unit short and Co the cost of one unit over. With Cu = 8 and Co = 2 the ratio is 0.8, so order the 80th percentile, not the mean. A quantile gradient-boosted model gives that directly:

In [2]:
Cu, Co = 8.0, 2.0
ratio = Cu / (Cu + Co)
q_model = HistGradientBoostingRegressor(loss="quantile", quantile=ratio, max_iter=200, learning_rate=0.08, random_state=0).fit(train[features], train.sales)
order_mean = gbm.predict(test[features])
order_q = q_model.predict(test[features])
demand = test.sales.values

def cost(order): return (Cu * np.maximum(demand - order, 0) + Co * np.maximum(order - demand, 0)).sum()
print(f"critical ratio {ratio:.2f}")
print(f"order the mean forecast    : cost {cost(order_mean):9,.0f} | share of weeks short {np.mean(demand > order_mean):.0%}")
print(f"order the {ratio:.0%} quantile      : cost {cost(order_q):9,.0f} | share of weeks short {np.mean(demand > order_q):.0%}")

critical ratio 0.80
order the mean forecast    : cost    33,113 | share of weeks short 48%
order the 80% quantile      : cost    24,740 | share of weeks short 22%


Ordering the mean leaves the SKU short about half the weeks, which is expensive when a shortage costs four times an overage. Ordering the 80th percentile is short about one week in five and has lower total cost.

#### Sales are not demand

When an item sells out, recorded sales stop at the stock level, so the history understates what customers wanted. A model trained on raw sales learns the cap and under-orders next time, which creates more stockouts. Fixes: flag stockout periods and treat those observations as censored, impute demand for them, or use availability as a feature.

In [3]:
rng = np.random.default_rng(1)
true_demand = rng.poisson(20, 10000)
stock = 18
sales = np.minimum(true_demand, stock)
print(f"true mean demand {true_demand.mean():.1f} | mean recorded sales {sales.mean():.1f} | weeks sold out {np.mean(true_demand >= stock):.0%}")

true mean demand 20.0 | mean recorded sales 17.1 | weeks sold out 70%


#### Time-series cross-validation

Shuffling trains on future, tests on past — leakage. Fix: walk-forward validation — expanding/rolling window, train strictly precedes validation fold.

In [1]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
for i, (tr, te) in enumerate(TimeSeriesSplit(n_splits=4).split(np.arange(20))):
    print(f"fold {i}: train weeks {tr.min()}-{tr.max()} | test weeks {te.min()}-{te.max()}")

fold 0: train weeks 0-3 | test weeks 4-7
fold 1: train weeks 0-7 | test weeks 8-11
fold 2: train weeks 0-11 | test weeks 12-15
fold 3: train weeks 0-15 | test weeks 16-19


#### Forecast metrics: MAPE, WAPE and MASE

MAPE: mean |err/actual| — undefined at 0, asymmetric (over-forecasts penalized less). WAPE=Σ|err|/Σactual — volume-weighted, robust to small items. MASE: MAE ÷ naive-forecast MAE — scale-free, <1 beats naive. Also track bias (mean error) and pinball loss for quantiles.

In [2]:
import numpy as np
actual = np.array([100, 80, 120, 2, 1, 3]); forecast = np.array([105, 78, 118, 4, 2, 6])
mape = np.mean(np.abs(actual - forecast) / actual); wape = np.abs(actual - forecast).sum() / actual.sum()
print(f"MAPE {mape:.0%} (dominated by the three tiny SKUs) | WAPE {wape:.1%} (weighted by volume)")

MAPE 52% (dominated by the three tiny SKUs) | WAPE 4.9% (weighted by volume)


### Blogs

- **Zalando.** Zalando's inventory system produces probabilistic demand forecasts weekly for about 5 million SKUs (article and color) over a 12-week horizon, using LightGBM with Nixtla's MLForecast on 2.5 years of sales and availability history, in under 2 hours. The forecasts are inputs to a recommendation engine that also uses lead times, user and item economics and current stock, and runs Monte Carlo simulations with gradient-free optimizers to minimize storage, lost-sales, overstock, operations and inbound costs under uncertainty. This is the same structure as the demo above: one global gradient-boosted model, a forecast distribution, and a cost-based decision on top. [Building a dynamic inventory optimisation system: A deep dive](https://engineering.zalando.com/posts/2025/06/inventory-optimisation-system.html)

#### Pitfalls

- Per-SKU models that cannot cover new or low-volume items.
- Judging a forecast by MAPE on low-volume items.
- Ordering to the mean forecast when stockout and overstock costs differ.
- Training on sales during stockouts as if it were demand.
- Random k-fold validation, which leaks the future: always split by time.